In [1]:
!python3 -V

Python 3.12.13


# Setup

## Dependencies

In [2]:
!pip install transformer_lens gradio datasets huggingface_hub plotly pandas

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 105.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 1.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 78.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 82.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 3.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import transformer_lens
from transformer_lens import HookedTransformer, utils
import torch
import numpy as np
import gradio as gr
import pprint
import json
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import HfApi
from IPython.display import HTML
from functools import partial
import tqdm.notebook as tqdm
import plotly.express as px
import pandas as pd

/tmp/ipykernel_1357/1824573690.py:2: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens import HookedTransformer, utils


## Defining the Autoencoder

Здесь задается sparse autoencoder (SAE), который раскладывает MLP-активацию `x` в разреженную сумму learned features.

Ключевая идея: hidden-слой шире входа в `dict_mult` раз, поэтому SAE ищет overcomplete basis - больше признаков, чем исходных нейронных координат. L1-штраф заставляет одновременно активироваться только малую часть признаков.

In [4]:
cfg = {
    "seed": 49,
    "batch_size": 4096,
    "buffer_mult": 384,
    "lr": 1e-4,
    "num_tokens": int(2e9),
    "l1_coeff": 3e-4,
    "beta1": 0.9,
    "beta2": 0.99,
    "dict_mult": 8,
    "seq_len": 128,
    "d_mlp": 2048,
    "enc_dtype":"fp32",
    "remove_rare_dir": False,
}
cfg["model_batch_size"] = 64
cfg["buffer_size"] = cfg["batch_size"] * cfg["buffer_mult"]
cfg["buffer_batches"] = cfg["buffer_size"] // cfg["seq_len"]

In [5]:
DTYPES = {"fp32": torch.float32, "fp16": torch.float16, "bf16": torch.bfloat16}
class AutoEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d_hidden = cfg["d_mlp"] * cfg["dict_mult"]
        d_mlp = cfg["d_mlp"]
        l1_coeff = cfg["l1_coeff"]
        dtype = DTYPES[cfg["enc_dtype"]]
        torch.manual_seed(cfg["seed"])
        self.W_enc = nn.Parameter(torch.nn.init.kaiming_uniform_(torch.empty(d_mlp, d_hidden, dtype=dtype)))
        self.W_dec = nn.Parameter(torch.nn.init.kaiming_uniform_(torch.empty(d_hidden, d_mlp, dtype=dtype)))
        self.b_enc = nn.Parameter(torch.zeros(d_hidden, dtype=dtype))
        self.b_dec = nn.Parameter(torch.zeros(d_mlp, dtype=dtype))

        self.W_dec.data[:] = self.W_dec / self.W_dec.norm(dim=-1, keepdim=True)

        self.d_hidden = d_hidden
        self.l1_coeff = l1_coeff

        self.to("cuda")

    def forward(self, x):
        x_cent = x - self.b_dec
        acts = F.relu(x_cent @ self.W_enc + self.b_enc)
        x_reconstruct = acts @ self.W_dec + self.b_dec
        l2_loss = (x_reconstruct.float() - x.float()).pow(2).sum(-1).mean(0)
        l1_loss = self.l1_coeff * (acts.float().abs().sum())
        loss = l2_loss + l1_loss
        return loss, x_reconstruct, acts, l2_loss, l1_loss

    @torch.no_grad()
    def remove_parallel_component_of_grads(self):
        # TODO: разобраться, вроде здесь про W_dec - вектор фичи
        W_dec_normed = self.W_dec / self.W_dec.norm(dim=-1, keepdim=True)
        W_dec_grad_proj = (self.W_dec.grad * W_dec_normed).sum(-1, keepdim=True) * W_dec_normed
        self.W_dec.grad -= W_dec_grad_proj

    # def get_version(self):
    #     return 1+max([int(file.name.split(".")[0]) for file in list(SAVE_DIR.iterdir()) if "pt" in str(file)])

    # def save(self):
    #     version = self.get_version()
    #     torch.save(self.state_dict(), SAVE_DIR/(str(version)+".pt"))
    #     with open(SAVE_DIR/(str(version)+"_cfg.json"), "w") as f:
    #         json.dump(cfg, f)
    #     print("Saved as version", version)

    # def load(cls, version):
    #     cfg = (json.load(open(SAVE_DIR/(str(version)+"_cfg.json"), "r")))
    #     pprint.pprint(cfg)
    #     self = cls(cfg=cfg)
    #     self.load_state_dict(torch.load(SAVE_DIR/(str(version)+".pt")))
    #     return self

    @classmethod
    def load_from_hf(cls, version):
        """
        Loads the saved autoencoder from HuggingFace.

        Version is expected to be an int, or "run1" or "run2"

        version 25 is the final checkpoint of the first autoencoder run,
        version 47 is the final checkpoint of the second autoencoder run.
        """
        if version=="run1":
            version = 25
        elif version=="run2":
            version = 47

        cfg = utils.download_file_from_hf("NeelNanda/sparse_autoencoder", f"{version}_cfg.json")
        pprint.pprint(cfg)
        self = cls(cfg=cfg)
        self.load_state_dict(utils.download_file_from_hf("NeelNanda/sparse_autoencoder", f"{version}.pt", force_is_torch=True))
        return self


## Utils

### Get Reconstruction Loss

Эти хуки проверяют, действительно ли SAE сохранил функциональное поведение MLP. Ноутбук прогоняет transformer как обычно, затем в точке `blocks.0.mlp.hook_post` заменяет настоящие MLP-активации реконструкцией SAE и сравнивает loss.

Если loss почти не ухудшается относительно исходной модели и намного лучше, чем при занулении MLP, значит найденные признаки не просто красивые визуализации: через них можно приблизительно восстановить причинно важный сигнал слоя.

`blocks.0.mlp.hook_post` - 1 трансформерный блок, внутрений слой активации размерности `d_model x 4` 

In [6]:
def replacement_hook(mlp_post, hook, encoder):
    """Replace MLP post-activations with their SAE reconstruction.

    This hook is attached to `blocks.0.mlp.hook_post`. `mlp_post` has
    shape `[batch, seq_len, d_mlp]`; the SAE returns a reconstruction of
    the same activation tensor, which is then passed back into the model.
    """
    mlp_post_reconstr = encoder(mlp_post)[1]
    return mlp_post_reconstr

def mean_ablate_hook(mlp_post, hook):
    """Replace every MLP post-activation with the batch-and-position mean.

    This is an ablation baseline: the MLP keeps its average activation
    vector, but loses token-specific information from individual positions.
    """
    mlp_post[:] = mlp_post.mean([0, 1])
    return mlp_post

def zero_ablate_hook(mlp_post, hook):
    """Zero out all MLP post-activations at the hooked layer.

    This provides a strong ablation baseline for measuring how much useful
    signal the layer contributes compared with the SAE reconstruction.
    """
    mlp_post[:] = 0.
    return mlp_post

@torch.no_grad()
def get_recons_loss(num_batches=5, local_encoder=None):
    """Evaluate how well an SAE reconstruction preserves model behavior.

    For random token batches, this compares the original model loss, the
    loss after replacing `blocks.0.mlp.hook_post` with SAE reconstruction,
    and the loss after zero-ablating that activation. The reconstruction
    score is `(zero_abl_loss - recons_loss) / (zero_abl_loss - loss)`.

    Args:
        num_batches: Number of random model batches to average over.
        local_encoder: SAE to evaluate. Defaults to the global `encoder`.

    Returns:
        Tuple `(score, loss, recons_loss, zero_abl_loss)` with averaged
        losses and the fraction of zero-ablation damage recovered by SAE.
    """
    if local_encoder is None:
        local_encoder = encoder
    loss_list = []
    for i in range(num_batches):
        tokens = all_tokens[torch.randperm(len(all_tokens))[:cfg["model_batch_size"]]]
        loss = model(tokens, return_type="loss")
        recons_loss = model.run_with_hooks(tokens, return_type="loss", fwd_hooks=[(utils.get_act_name("post", 0), partial(replacement_hook, encoder=local_encoder))])
        # mean_abl_loss = model.run_with_hooks(tokens, return_type="loss", fwd_hooks=[(utils.get_act_name("post", 0), mean_ablate_hook)])
        zero_abl_loss = model.run_with_hooks(tokens, return_type="loss", fwd_hooks=[(utils.get_act_name("post", 0), zero_ablate_hook)])
        loss_list.append((loss, recons_loss, zero_abl_loss))
    losses = torch.tensor(loss_list)
    loss, recons_loss, zero_abl_loss = losses.mean(0).tolist()

    print(f"loss: {loss:.4f}, recons_loss: {recons_loss:.4f}, zero_abl_loss: {zero_abl_loss:.4f}")
    score = ((zero_abl_loss - recons_loss)/(zero_abl_loss - loss))
    print(f"Reconstruction Score: {score:.2%}")
    # print(f"{((zero_abl_loss - mean_abl_loss)/(zero_abl_loss - loss)).item():.2%}")
    return score, loss, recons_loss, zero_abl_loss

### Get Frequencies

`get_freqs` измеряет плотность каждого SAE-признака: на какой доле токенов его активация больше нуля. Это соответствует feature density histograms из статьи.

Разреженность важна потому, что monosemantic feature обычно включается только в специфических контекстах. Очень редкие или мертвые признаки отдельно анализируются ниже: часть из них может быть артефактом обучения, а не настоящей смысловой фичей модели.

In [7]:
# Frequency
@torch.no_grad()
def get_freqs(num_batches=25, local_encoder=None):
    """Estimate activation frequency for every SAE feature.

    Randomly samples token batches, caches `blocks.0.mlp.hook_post`, encodes
    those MLP activations with the SAE, and counts how often each hidden
    feature is active (`hidden > 0`). The returned vector has length
    `local_encoder.d_hidden`; each entry is the fraction of sampled token
    positions on which that SAE feature fired.

    Args:
        num_batches: Number of random model batches to sample.
        local_encoder: SAE to evaluate. Defaults to the global `encoder`.

    Returns:
        Tensor of shape `[d_hidden]` with per-feature activation frequencies.
        Features with frequency exactly zero over this sample are counted as
        dead features and reported via `Num dead`.
    """
    if local_encoder is None:
        local_encoder = encoder
    act_freq_scores = torch.zeros(local_encoder.d_hidden, dtype=torch.float32).cuda()
    total = 0
    for i in tqdm.trange(num_batches):
        tokens = all_tokens[torch.randperm(len(all_tokens))[:cfg["model_batch_size"]]]

        _, cache = model.run_with_cache(tokens, stop_at_layer=1, names_filter=utils.get_act_name("post", 0))
        mlp_acts = cache[utils.get_act_name("post", 0)]
        mlp_acts = mlp_acts.reshape(-1, d_mlp)

        hidden = local_encoder(mlp_acts)[2]

        act_freq_scores += (hidden > 0).sum(0)
        total+=hidden.shape[0]
    act_freq_scores /= total
    num_dead = (act_freq_scores==0).float().mean()
    print("Num dead", num_dead)
    return act_freq_scores

## Visualise Feature Utils

In [8]:
from html import escape
import colorsys

from IPython.display import display

SPACE = "·"
NEWLINE="↩"
TAB = "→"

def create_html(strings, values, max_value=None, saturation=0.5, allow_different_length=False, return_string=False):
    # escape strings to deal with tabs, newlines, etc.
    escaped_strings = [escape(s, quote=True) for s in strings]
    processed_strings = [
        s.replace("\n", f"{NEWLINE}<br/>").replace("\t", f"{TAB}&emsp;").replace(" ", "&nbsp;")
        for s in escaped_strings
    ]

    if isinstance(values, torch.Tensor) and len(values.shape)>1:
        values = values.flatten().tolist()

    if not allow_different_length:
        assert len(processed_strings) == len(values)

    # scale values
    if max_value is None:
        max_value = max(max(values), -min(values))+1e-3
    scaled_values = [v / max_value * saturation for v in values]

    # create html
    html = ""
    for i, s in enumerate(processed_strings):
        if i<len(scaled_values):
            v = scaled_values[i]
        else:
            v = 0
        if v < 0:
            hue = 0  # hue for red in HSV
        else:
            hue = 0.66  # hue for blue in HSV
        rgb_color = colorsys.hsv_to_rgb(
            hue, v, 1
        )  # hsv color with hue 0.66 (blue), saturation as v, value 1
        hex_color = "#%02x%02x%02x" % (
            int(rgb_color[0] * 255),
            int(rgb_color[1] * 255),
            int(rgb_color[2] * 255),
        )
        html += f'<span style="background-color: {hex_color}; border: 1px solid lightgray; font-size: 16px; border-radius: 3px;">{s}</span>'
    if return_string:
        return html
    else:
        display(HTML(html))

def basic_feature_vis(text, feature_index, max_val=0):
    feature_in = encoder.W_enc[:, feature_index]
    feature_bias = encoder.b_enc[feature_index]
    _, cache = model.run_with_cache(text, stop_at_layer=1, names_filter=utils.get_act_name("post", 0))
    mlp_acts = cache[utils.get_act_name("post", 0)][0]
    feature_acts = F.relu((mlp_acts - encoder.b_dec) @ feature_in + feature_bias)
    if max_val==0:
        max_val = max(1e-7, feature_acts.max().item())
        # print(max_val)
    # if min_val==0:
    #     min_val = min(-1e-7, feature_acts.min().item())
    return basic_token_vis_make_str(text, feature_acts, max_val)
def basic_token_vis_make_str(strings, values, max_val=None):
    if not isinstance(strings, list):
        strings = model.to_str_tokens(strings)
    values = utils.to_numpy(values)
    if max_val is None:
        max_val = values.max()
    # if min_val is None:
    #     min_val = values.min()
    header_string = f"<h4>Max Range <b>{values.max():.4f}</b> Min Range: <b>{values.min():.4f}</b></h4>"
    header_string += f"<h4>Set Max Range <b>{max_val:.4f}</b></h4>"
    # values[values>0] = values[values>0]/ma|x_val
    # values[values<0] = values[values<0]/abs(min_val)
    body_string = create_html(strings, values, max_value=max_val, return_string=True)
    return header_string + body_string
# display(HTML(basic_token_vis_make_str(tokens[0, :10], mlp_acts[0, :10, 7], 0.1)))
# # %%
# The `with gr.Blocks() as demo:` syntax just creates a variable called demo containing all these components
import gradio as gr
try:
    demos[0].close()
except:
    pass
demos = [None]
def make_feature_vis_gradio(feature_id, starting_text=None, batch=None, pos=None):
    if starting_text is None:
        starting_text = model.to_string(all_tokens[batch, 1:pos+1])
    try:
        demos[0].close()
    except:
        pass
    with gr.Blocks() as demo:
        gr.HTML(value=f"Hacky Interactive Neuroscope for gelu-1l")
        # The input elements
        with gr.Row():
            with gr.Column():
                text = gr.Textbox(label="Text", value=starting_text)
                # Precision=0 makes it an int, otherwise it's a float
                # Value sets the initial default value
                feature_index = gr.Number(
                    label="Feature Index", value=feature_id, precision=0
                )
                # # If empty, these two map to None
                max_val = gr.Number(label="Max Value", value=None)
                # min_val = gr.Number(label="Min Value", value=None)
                inputs = [text, feature_index, max_val]
        with gr.Row():
            with gr.Column():
                # The output element
                out = gr.HTML(label="Neuron Acts", value=basic_feature_vis(starting_text, feature_id))
        for inp in inputs:
            inp.change(basic_feature_vis, inputs, out)
    demo.launch(share=True)
    demos[0] = demo

### Inspecting Top Logits

In [9]:
SPACE = "·"
NEWLINE="↩"
TAB = "→"
def process_token(s):
    if isinstance(s, torch.Tensor):
        s = s.item()
    if isinstance(s, np.int64):
        s = s.item()
    if isinstance(s, int):
        s = model.to_string(s)
    s = s.replace(" ", SPACE)
    s = s.replace("\n", NEWLINE+"\n")
    s = s.replace("\t", TAB)
    return s

def process_tokens(l):
    if isinstance(l, str):
        l = model.to_str_tokens(l)
    elif isinstance(l, torch.Tensor) and len(l.shape)>1:
        l = l.squeeze(0)
    return [process_token(s) for s in l]

def process_tokens_index(l):
    if isinstance(l, str):
        l = model.to_str_tokens(l)
    elif isinstance(l, torch.Tensor) and len(l.shape)>1:
        l = l.squeeze(0)
    return [f"{process_token(s)}/{i}" for i,s in enumerate(l)]

def create_vocab_df(logit_vec, make_probs=False, full_vocab=None):
    if full_vocab is None:
        full_vocab = process_tokens(model.to_str_tokens(torch.arange(model.cfg.d_vocab)))
    vocab_df = pd.DataFrame({"token": full_vocab, "logit": utils.to_numpy(logit_vec)})
    if make_probs:
        vocab_df["log_prob"] = utils.to_numpy(logit_vec.log_softmax(dim=-1))
        vocab_df["prob"] = utils.to_numpy(logit_vec.softmax(dim=-1))
    return vocab_df.sort_values("logit", ascending=False)

### Make Token DataFrame

In [10]:
def list_flatten(nested_list):
    return [x for y in nested_list for x in y]
def make_token_df(tokens, len_prefix=5, len_suffix=1):
    str_tokens = [process_tokens(model.to_str_tokens(t)) for t in tokens]
    unique_token = [[f"{s}/{i}" for i, s in enumerate(str_tok)] for str_tok in str_tokens]

    context = []
    batch = []
    pos = []
    label = []
    for b in range(tokens.shape[0]):
        # context.append([])
        # batch.append([])
        # pos.append([])
        # label.append([])
        for p in range(tokens.shape[1]):
            prefix = "".join(str_tokens[b][max(0, p-len_prefix):p])
            if p==tokens.shape[1]-1:
                suffix = ""
            else:
                suffix = "".join(str_tokens[b][p+1:min(tokens.shape[1]-1, p+1+len_suffix)])
            current = str_tokens[b][p]
            context.append(f"{prefix}|{current}|{suffix}")
            batch.append(b)
            pos.append(p)
            label.append(f"{b}/{p}")
    # print(len(batch), len(pos), len(context), len(label))
    return pd.DataFrame(dict(
        str_tokens=list_flatten(str_tokens),
        unique_token=list_flatten(unique_token),
        context=context,
        batch=batch,
        pos=pos,
        label=label,
    ))

## Loading the Model

<!-- codex-ru-explainer -->

**Русский комментарий.** Загружается маленький one-layer transformer `gelu-1l` из TransformerLens. Это удобная учебная модель: MLP всего один, поэтому downstream-эффект SAE-признака можно напрямую смотреть через финальные logits.

In [11]:
model = HookedTransformer.from_pretrained("gelu-1l").to(DTYPES[cfg["enc_dtype"]])
n_layers = model.cfg.n_layers
d_model = model.cfg.d_model
n_heads = model.cfg.n_heads
d_head = model.cfg.d_head
d_mlp = model.cfg.d_mlp
d_vocab = model.cfg.d_vocab

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

./model_final.pth:   0%|          | 0.00/213M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.04M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

Loaded pretrained model gelu-1l into HookedTransformer
Changing model dtype to torch.float32


## Loading Data

In [12]:
data = load_dataset("NeelNanda/c4-code-20k", split="train")
tokenized_data = utils.tokenize_and_concatenate(data, model.tokenizer, max_length=128)
tokenized_data = tokenized_data.shuffle(42)
all_tokens = tokenized_data["tokens"]

README.md:   0%|          | 0.00/754 [00:00<?, ?B/s]

data/train-00000-of-00001-97684e149eb4d6(…):   0%|          | 0.00/42.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map (num_proc=10):   0%|          | 0/20000 [00:00<?, ? examples/s]

## Training a Local SAE

The original tutorial loads a pretrained SAE from `NeelNanda/sparse_autoencoder`. The cells below train a small local SAE on the same `gelu-1l` MLP activations, then keep it in `trained_encoder` for comparison.


Этот блок делает именно обучение SAE: на каждом шаге мы берем батч текстов, прогоняем `gelu-1l`, достаем activations из `blocks.0.mlp.hook_post`, и учим автоэнкодер восстанавливать эти activations с L1-штрафом на скрытые признаки. Параметры ниже уменьшены относительно большого запуска Neel Nanda, чтобы эксперимент можно было воспроизвести в ноутбуке.

In [30]:
local_sae_cfg = cfg.copy()
local_sae_cfg.update({
    "seed": 123,
    "dict_mult": 8,          # smaller than NeelNanda's 8x run, so notebook training is practical
    "batch_size": 2048,      # number of MLP activation vectors per optimizer step
    "lr": 1e-4,
    "l1_coeff": 3e-4,
    "enc_dtype": "fp32",
})

TRAIN_LOCAL_SAE = True       # set False if you only want to use the pretrained HF SAE
LOCAL_SAE_STEPS = 1000        # increase to a few thousand for visibly better reconstruction
LOCAL_SAE_LOG_EVERY = 25
LOCAL_SAE_PATH = "local_sae_gelu_1l.pt"

In [31]:
@torch.no_grad()
def sample_mlp_post_acts(num_activation_vectors, token_source=None):
    if token_source is None:
        token_source = all_tokens

    seq_count = int(np.ceil(num_activation_vectors / cfg["seq_len"]))
    token_idxs = torch.randperm(len(token_source))[:seq_count]
    tokens = token_source[token_idxs]

    _, cache = model.run_with_cache(
        tokens,
        stop_at_layer=1,
        names_filter=utils.get_act_name("post", 0),
    )
    mlp_acts = cache[utils.get_act_name("post", 0)].reshape(-1, d_mlp)
    return mlp_acts[:num_activation_vectors].detach()


def normalize_decoder_weights(local_encoder):
    with torch.no_grad():
        local_encoder.W_dec.data /= local_encoder.W_dec.data.norm(dim=-1, keepdim=True).clamp_min(1e-8)


def train_local_sae(local_cfg, num_steps=300, log_every=25):
    local_encoder = AutoEncoder(local_cfg)
    optimizer = torch.optim.Adam(
        local_encoder.parameters(),
        lr=local_cfg["lr"],
        betas=(local_cfg["beta1"], local_cfg["beta2"]),
    )
    history = []
    progress = tqdm.trange(num_steps)

    for step in progress:
        x = sample_mlp_post_acts(local_cfg["batch_size"])
        loss, x_reconstruct, acts, l2_loss, l1_loss = local_encoder(x)

        loss.backward()
        local_encoder.remove_parallel_component_of_grads()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        normalize_decoder_weights(local_encoder)

        if step == 0 or (step + 1) % log_every == 0 or step + 1 == num_steps:
            with torch.no_grad():
                mse = (x_reconstruct.float() - x.float()).pow(2).mean().item()
                l0 = (acts > 0).float().sum(dim=-1).mean().item()
                density = (acts > 0).float().mean().item()
            row = {
                "step": step + 1,
                "loss": loss.item(),
                "l2_loss": l2_loss.item(),
                "l1_loss": l1_loss.item(),
                "mse_per_activation": mse,
                "mean_l0": l0,
                "density": density,
            }
            history.append(row)
            progress.set_postfix({"loss": f"{loss.item():.2f}", "mse": f"{mse:.4f}", "l0": f"{l0:.1f}"})

    return local_encoder, pd.DataFrame(history)


if TRAIN_LOCAL_SAE:
    trained_encoder, local_sae_history = train_local_sae(
        local_sae_cfg,
        num_steps=LOCAL_SAE_STEPS,
        log_every=LOCAL_SAE_LOG_EVERY,
    )
    torch.save({"cfg": local_sae_cfg, "state_dict": trained_encoder.state_dict()}, LOCAL_SAE_PATH)
else:
    trained_encoder = None
    local_sae_history = pd.DataFrame()

local_sae_history.tail()

  0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

Training curves are noisy because each step samples fresh activations. The most useful quick diagnostics are `mse_per_activation` going down and `mean_l0` staying much smaller than the full dictionary size.


Для настоящего качества нужно больше шагов и обычно больше dictionary size. Но даже короткий запуск полезен: он показывает полный pipeline обучения, а не только загрузку готовых весов.

In [15]:
if not local_sae_history.empty:
    px.line(
        local_sae_history,
        x="step",
        y=["mse_per_activation", "density"],
        title="Local SAE training diagnostics",
    )

# Analysis

## Loading the Autoencoder

There are two runs on separate random seeds, along with a bunch of intermediate checkpoints

Здесь не обучается новый autoencoder. Вместо этого скачивается уже обученный SAE из `NeelNanda/sparse_autoencoder`: `run1` или `run2`. Поэтому ноутбук - replication/tutorial для анализа готового dictionary learning результата, а не полный training pipeline.

In [16]:
auto_encoder_run = "run1" # @param ["run1", "run2"]
encoder = AutoEncoder.load_from_hf(auto_encoder_run)

25_cfg.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

{'batch_size': 4096,
 'beta1': 0.9,
 'beta2': 0.99,
 'buffer_batches': 12288,
 'buffer_mult': 384,
 'buffer_size': 1572864,
 'd_mlp': 2048,
 'dict_mult': 8,
 'enc_dtype': 'fp32',
 'l1_coeff': 0.0003,
 'lr': 0.0001,
 'model_batch_size': 512,
 'num_tokens': 2000000000,
 'seed': 52,
 'seq_len': 128}


./25.pt:   0%|          | 0.00/269M [00:00<?, ?B/s]

## Comparing Local SAE With `NeelNanda/sparse_autoencoder`

We compare the freshly trained SAE and the pretrained reference on the same held-out activation batches. This keeps the comparison apples-to-apples even though the local SAE is deliberately smaller and trained for far fewer steps.

Метрики ниже разделяют два вопроса: насколько хорошо SAE восстанавливает сами MLP activations (`mse_per_activation`, `l2_loss`) и насколько восстановление сохраняет поведение модели (`reconstruction_score`). Pretrained SAE должен быть лучше: он обучался дольше и с большим словарем.

In [17]:
@torch.no_grad()
def sae_activation_metrics(local_encoder, num_batches=5, batch_size=2048):
    rows = []
    for _ in tqdm.trange(num_batches):
        x = sample_mlp_post_acts(batch_size)
        _, x_reconstruct, acts, l2_loss, l1_loss = local_encoder(x)
        rows.append({
            "mse_per_activation": (x_reconstruct.float() - x.float()).pow(2).mean().item(),
            "l2_loss": l2_loss.item(),
            "l1_loss": l1_loss.item(),
            "mean_l0": (acts > 0).float().sum(dim=-1).mean().item(),
            "density": (acts > 0).float().mean().item(),
            "dead_feature_frac_on_eval_batches": ((acts > 0).float().sum(dim=0) == 0).float().mean().item(),
        })
    return pd.DataFrame(rows).mean().to_dict()


def compare_saes(reference_encoder, candidate_encoder=None):
    rows = []
    for name, local_encoder in [
        ("NeelNanda/sparse_autoencoder", reference_encoder),
        ("local_trained_sae", candidate_encoder),
    ]:
        if local_encoder is None:
            continue

        row = {"sae": name, "d_hidden": local_encoder.d_hidden}
        row.update(sae_activation_metrics(local_encoder, num_batches=5, batch_size=2048))
        score, loss, recons_loss, zero_abl_loss = get_recons_loss(num_batches=3, local_encoder=local_encoder)
        row.update({
            "model_loss": loss,
            "recons_loss": recons_loss,
            "zero_ablation_loss": zero_abl_loss,
            "reconstruction_score": score,
        })
        rows.append(row)

    return pd.DataFrame(rows)


sae_comparison_df = compare_saes(encoder, globals().get("trained_encoder", None))
sae_comparison_df.style.format({
    "mse_per_activation": "{:.5f}",
    "l2_loss": "{:.3f}",
    "l1_loss": "{:.3f}",
    "mean_l0": "{:.2f}",
    "density": "{:.5f}",
    "dead_feature_frac_on_eval_batches": "{:.2%}",
    "model_loss": "{:.3f}",
    "recons_loss": "{:.3f}",
    "zero_ablation_loss": "{:.3f}",
    "reconstruction_score": "{:.2%}",
})

  0%|          | 0/5 [00:00<?, ?it/s]

loss: 3.2059, recons_loss: 3.7018, zero_abl_loss: 8.7397
Reconstruction Score: 91.04%


  0%|          | 0/5 [00:00<?, ?it/s]

loss: 3.2239, recons_loss: 5.8058, zero_abl_loss: 8.7511
Reconstruction Score: 53.29%


,sae,d_hidden,mse_per_activation,l2_loss,l1_loss,mean_l0,density,dead_feature_frac_on_eval_batches,model_loss,recons_loss,zero_ablation_loss,reconstruction_score
0,NeelNanda/sparse_autoencoder,16384,0.01568,32.120,11.174,139.02,0.00849,62.16%,3.206,3.702,8.740,91.04%
1,local_trained_sae,8192,0.05919,121.224,19.720,111.32,0.01359,9.89%,3.224,5.806,8.751,53.29%


## Using the Autoencoder

We run the model and replace the MLP activations with those reconstructed from the autoencoder, and get 91% loss recovered

In [18]:
_ = get_recons_loss(num_batches=5, local_encoder=encoder)

loss: 3.2484, recons_loss: 3.7341, zero_abl_loss: 8.7699
Reconstruction Score: 91.20%


## Rare Features Are All The Same

Этот блок повторяет наблюдение из статьи про ультра-редкие признаки. Сначала строится histogram log-frequency, затем признаки с `freq < 1e-4` сравниваются с усредненным направлением rare features.

Если почти все редкие признаки имеют высокий cosine similarity с одним средним направлением, это подозрительно: они похожи не на разные содержательные features, а на общий артефакт SAE.

For each feature we can get the frequency at which it's non-zero (per token, averaged across a bunch of batches), and plot a histogram

In [19]:
freqs = get_freqs(num_batches = 50, local_encoder = encoder)

  0%|          | 0/50 [00:00<?, ?it/s]

Num dead tensor(0., device='cuda:0')


In [29]:
# Add 1e-6.5 so that dead features show up as log_freq -6.5
log_freq = (freqs + 10**-6.5).log10()
px.histogram(utils.to_numpy(log_freq), title="Log Frequency of Features", histnorm='percent')

We see that it's clearly bimodal! Let's define rare features as those with freq < 1e-4, and look at the cosine sim of each feature with the average rare feature - we see that almost all rare features correspond to this feature!

In [21]:
is_rare = freqs < 1e-4
rare_enc = encoder.W_enc[:, is_rare]
rare_mean = rare_enc.mean(-1)
px.histogram(utils.to_numpy(rare_mean @ encoder.W_enc / rare_mean.norm() / encoder.W_enc.norm(dim=0)), title="Cosine Sim with Ave Rare Feature", color=utils.to_numpy(is_rare), labels={"color": "is_rare", "count": "percent", "value": "cosine_sim"}, marginal="box", histnorm="percent", barmode='overlay')

## Interpreting A Feature

<!-- codex-ru-explainer -->

**Русский комментарий.** Дальше ноутбук показывает стандартный workflow интерпретации одного SAE-признака: выбрать feature id, найти токены/контексты с максимальной activation, вручную проверить гипотезу интерактивной визуализацией и затем посмотреть, какие logits усиливает decoder direction этого признака.

Let's go and investigate a non rare feature, feature 7

In [22]:
feature_id = 7 # @param {type:"number"}
batch_size = 128 # @param {type:"number"}

print(f"Feature freq: {freqs[7].item():.4f}")

Feature freq: 0.0030


Let's run the model on some text and then use the autoencoder to process the MLP activations

In [23]:
tokens = all_tokens[:batch_size]
_, cache = model.run_with_cache(tokens, stop_at_layer=1, names_filter=utils.get_act_name("post", 0))
mlp_acts = cache[utils.get_act_name("post", 0)]
mlp_acts_flattened = mlp_acts.reshape(-1, cfg["d_mlp"])
loss, x_reconstruct, hidden_acts, l2_loss, l1_loss = encoder(mlp_acts_flattened)
# This is equivalent to:
# hidden_acts = F.relu((mlp_acts_flattened - encoder.b_dec) @ encoder.W_enc + encoder.b_enc)
print("hidden_acts.shape", hidden_acts.shape)

hidden_acts.shape torch.Size([16384, 16384])


We can now sort and display the top tokens, and we see that this feature activates on text like " and I" (ditto for other connectives and pronouns)! It seems interpretable!

**Aside:** Note on how to read the context column:

A line like "·himself·as·democratic·socialist·and|·he|·favors" means that the preceding 5 tokens are " himself as democratic socialist and", the current token is " he" and the next token is " favors".  · are spaces, ↩ is a newline.

This gets a bit confusing for this feature, since the pipe separators look a lot like a capital I


In [24]:
token_df = make_token_df(tokens)
token_df["feature"] = utils.to_numpy(hidden_acts[:, feature_id])
token_df.sort_values("feature", ascending=False).head(20).style.background_gradient("coolwarm")

,str_tokens,unique_token,context,batch,pos,label,feature
7989,·we,·we/53,ors·and·even·heroes·and|·we|·feel,62,53,62/53,1.722712
14648,·he,·he/56,·1971·and|·he|·registered,114,56,114/56,1.211562
6643,·we,·we/115,·anchor·[xx]·but|·we|·want,51,115,51/115,0.798183
1546,·I,·I/10,ing·experience–and·while|·I|·continue,12,10,12/10,0.550763
7618,·you,·you/66,"·family·law·issues,·then|·you|·came",59,66,59/66,0.408081
16361,·there,·there/105,"·simple,·honest·mistakes·and|·there|·""",127,105,127/105,0.388667
4630,·y,·y/22,"·Super·friendly,·affordable·and|·y|ummy",36,22,36/22,0.359096
1559,I,I/23,·amazing·platform·for·creators–|I|’,12,23,12/23,0.285587
7961,·people,·people/25,·rich·cultural·heritage·and·the|·people|·who,62,25,62/25,0.274128
13948,·there,·there/124,"·documentation·for·these,·but|·there|·are",108,124,108/124,0.272422


It's easy to misread evidence like the above, so it's useful to take some text and edit it and see how this changes the model's activations. Here's a hacky interactive tool to play around with some text.

In [25]:
model.cfg

HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'NTK_original_ctx_len': 8192,
 'act_fn': 'gelu',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': np.float64(8.0),
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'd_head': 64,
 'd_mlp': 2048,
 'd_model': 512,
 'd_vocab': 48262,
 'd_vocab_out': 48262,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': 'cuda',
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': False,
 'init_mode': 'gpt2',
 'init_weights': False,
 'initializer_range': np.float64(0.035355339059327376),
 'layer_norm_folding': False,
 'load_in_4bit': False,
 'model_name': 'GELU_1L512W_C4_Code',
 'n_ctx': 1024,
 'n_devices': 1,
 'n_heads': 8,
 'n_key_value_heads': None,
 'n_layers': 1,
 'n_p

In [26]:
s = "The 1899 Kentucky gubernatorial election was held on November 7, 1899. The Republican incumbent, William Bradley, was term-limited. The Democrats chose William Goebel. Republicans nominated William Taylor. Taylor won by a vote of 193,714 to 191,331. The vote was challenged on grounds of voter fraud, but the Board of Elections, though stocked with pro-Goebel members, certified the result. Democratic legislators began investigations, but before their committee could report, Goebel was shot by an unknown assassin (event pictured) on January 30, 1900. Democrats voided enough votes to swing the election to Goebel, Taylor was deposed, and Goebel was sworn into office on January 31. He died on February 3. The lieutenant governor of Kentucky, J. C. W. Beckham, became governor, and battled Taylor in court. Beckham won on appeal, and Taylor fled to Indiana, fearing arrest as an accomplice. The only persons convicted in connection with the killing were later pardoned; the assassin's identity remains a mystery"
t = model.to_tokens(s)
print(t)

tensor([[    1,   510,   338,    26,    27,    27, 17846,   306, 22621, 25060,
          5962,   369,  2828,   328,  4452,   813,    14,   338,    26,    27,
            27,    16,   380,  8558, 34001,    14,  7060, 27032,    14,   369,
          1283,    15, 15430,    16,   380, 10540,  9448,  7060,  3497,  2204,
           294,    16, 11564, 24709,  7060, 10978,    16, 10978,  1856,   407,
           248,  6101,   274,   338,    27,    21,    14,    25,    19,    22,
           282,   338,    27,    19,    14,    21,    21,    19,    16,   380,
          6101,   369, 14459,   328,  9643,   274, 23044,  8879,    14,   532,
           254,  5816,   274,   444,  9628,    14,  2101,  5574,   265,   343,
           355,    15,  6644,  2204,   294,  2672,    14, 17551,   254,   898,
            16,  9659, 35835,  3297, 13625,    14,   532,  1063,   615,  9106,
           807,  1280,    14,  3497,  2204,   294,   369,  4950,   407,   272,
          7011, 44280,   314,  7833, 34609,    11,  

In [27]:

starting_text = "Hero and I will head to Samantha and Mark's, then he and she will. Then I or you" # @param {type:"string"}
make_feature_vis_gradio(feature_id, starting_text)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c8b5acc727b2187d18.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


A final piece of evidence: This is a one layer model, so the neurons can only matter by directly impacting the final logits! We can directly look at how the decoder weights for this feature affect the logits, and see that it boosts `'ll`! This checks out, I and he'll etc is a common construction.

<!-- codex-ru-explainer -->

**Русский комментарий.** В однослойной модели влияние MLP-активации на следующий токен почти линейно читается через `W_out @ W_U`. Поэтому строка ниже берет decoder direction выбранного признака и переводит его в logit effects: какие токены становятся более/менее вероятными, когда этот feature активируется.

In [28]:
logit_effect = encoder.W_dec[feature_id] @ model.W_out[0] @ model.W_U
create_vocab_df(logit_effect).head(20).style.background_gradient("coolwarm")

,token,logit
1782,'ll,1.875001
6835,eding,1.558904
5440,·certainly,1.522172
3409,·hope,1.521111
29310,OULD,1.508625
4931,·wouldn,1.483282
44469,cheon,1.435624
7754,·definitely,1.397462
1606,·seem,1.394357
41304,·sincerely,1.363966
